Conor Murray Assignment 1

Question 1: Your task is to code a decision tree model training program in Python following the functional framework
provided below.
1. Function: createBranches(att)
▪ Purpose: find possible values (i.e., branches) of a given attribute
▪ Inputs: attribute name
▪ Outputs: list of branches
▪ Example: for the illustrative data set provided, createBranches(‘packetSize’)
should return [‘Small’,’Medium’,’Large’].
2. Function: divideByLabel(path)
▪ Purpose: find the breakdown of observations that fall under a given path in terms
of the class labels
▪ Inputs: path. A path is a collection of branches. For instance, in the example data
set provided, (df[‘source’] == ‘Asia’ & df[‘whiteListedIP’] == ‘Yes’) is a path that
leads to 3 observations. The breakdown of these observations is 3 Yes and 0 No.
▪ Outputs: a dictionary that consists of the frequency of each class label under this
path.
▪ Example: For the example above, the return should be {‘Y’:3, ‘N’:0}
3. Function: calcEntropy(aDict)
▪ Purpose: calculate the entropy of a given label breakdown
▪ Inputs: the dictionary output from divideByLabel
▪ Outputs: entropy score as a real number
▪ Example: calcEntropy({‘Y’:3, ‘N’:0}) should return 0.0.
4. Function: findSplitAtt(branch,attList)
▪ Purpose: find which attribute to use for split at each iteration
▪ Inputs: current path and a list of all attributes
▪ Outputs: the name of the attribute to use
5. Function: checkStopping(branch,attList)
▪ Purpose: check if a path growth needs to stop or not
▪ Inputs: a path and a list of attributes
▪ Outputs: 3 possible stopping conditions or a continue indicator
6. Function: main(df)
▪ Purpose: main program
▪ Inputs: dataset (Pandas dataframe)
▪ Outputs: a list of lists – each item in the outer list includes a complete path (from
the top of the tree to the leaf node).

In [16]:
import pandas as pd
import math
df = pd.read_csv('DT_coding_sample_training.csv')
labelCol = 'malicious'

In [49]:
def createBranches(att):
    branches = []
    for value in df[att]:
        if value not in branches:
            branches.append(value)
    return branches
print("createBranches('packetSize')", createBranches('packetSize'))

createBranches('packetSize') ['Small', 'Large', 'Medium']


In [39]:
def divideByLabel(path):
    mask = pd.Series(True, index=df.index)
    for i in range(len(path)):
        att = path[i][0]
        value = path[i][1]
        mask = mask & (df[att] == value)
    subset = df[mask]
    breakdown = {}
    for label in createBranches(labelCol):
        breakdown[label] = 0
    for label in subset[labelCol]:
        breakdown[label] = breakdown[label] + 1
    return breakdown
print("divideByLabel", divideByLabel([('source','Asia'), ('whiteListedIP','Yes')]))

divideByLabel {'Yes': 3, 'No': 0}


In [31]:
def calcEntropy(aDict):
    total = 0
    for label in aDict:
        total = total + aDict[label]
    if total == 0:
        return 0.0
    entropy = 0.0
    for label in aDict:
        count = aDict[label]
        if count > 0:
            p = count / total
            entropy = entropy - p * math.log(p, 2)
    return entropy
print("calcEntropy({'Yes':3, 'No':0})", calcEntropy({'Yes':3, 'No':0}))

calcEntropy({'Yes':3, 'No':0}) 0.0


In [20]:
def remainingAtts(path, attList):
    usedAtts = []
    for i in range(len(path)):
        usedAtts.append(path[i][0])
    remaining = []
    for i in range(len(attList)):
        att = attList[i]
        if att not in usedAtts:
            remaining.append(att)
    return remaining

In [38]:
def findSplitAtt(branch, attList):
    parentDict = divideByLabel(branch)
    parentEntropy = calcEntropy(parentDict)
    parentTotal = 0
    for label in parentDict:
        parentTotal = parentTotal + parentDict[label]
    candidates = remainingAtts(branch, attList)
    bestAtt = None
    bestGain = -1.0
    for i in range(len(candidates)):
        att = candidates[i]
        weightedEntropy = 0.0
        values = createBranches(att)
        for j in range(len(values)):
            value = values[j]
            childPath = branch + [(att, value)]
            childDict = divideByLabel(childPath)
            childTotal = 0
            for label in childDict:
                childTotal = childTotal + childDict[label]
            if childTotal > 0:
                weight = childTotal / parentTotal
                weightedEntropy = weightedEntropy + weight * calcEntropy(childDict)
        gain = parentEntropy - weightedEntropy
        if gain > bestGain:
            bestGain = gain
            bestAtt = att
    return bestAtt
print("findSplitAtt", findSplitAtt([], ['source','whiteListedIP','packetSize','appType']))

findSplitAtt source


In [48]:
def checkStopping(branch, attList):
    breakdown = divideByLabel(branch)
    total = 0
    for label in breakdown:
        total = total + breakdown[label]
    if total == 0:
        return 'noData'
    nonZeroLabels = 0
    for label in breakdown:
        if breakdown[label] > 0:
            nonZeroLabels = nonZeroLabels + 1
    if nonZeroLabels == 1:
        return 'pure'
    if len(remainingAtts(branch, attList)) == 0:
        return 'noAtts'
    return 'continue'
print("checkStopping(root)", checkStopping([], ['source','whiteListedIP','packetSize','appType']))
print("checkStopping(Asia, Yes)", checkStopping([('source','Asia'), ('whiteListedIP','Yes')], ['source','whiteListedIP','packetSize','appType']))
print("checkStopping(US, Yes, Small)", checkStopping([('source','US'), ('whiteListedIP','Yes'), ('packetSize','Small')], ['source','whiteListedIP','packetSize','appType']))

checkStopping(root) continue
checkStopping(Asia, Yes) pure
checkStopping(US, Yes, Small) noData


In [23]:
def majorityLabel(path):
    breakdown = divideByLabel(path)
    bestLabel = None
    bestCount = -1
    for label in breakdown:
        if breakdown[label] > bestCount:
            bestCount = breakdown[label]
            bestLabel = label
    return bestLabel

In [40]:
def main(df):
    attList = []
    for col in df.columns:
        if col != labelCol:
            attList.append(col)
    allPaths = []
    pathsToProcess = []
    pathsToProcess.append([])
    while len(pathsToProcess) > 0:
        currentPath = pathsToProcess[0]
        pathsToProcess = pathsToProcess[1:]
        status = checkStopping(currentPath, attList)
        if status == 'noData':
            continue
        if status == 'pure' or status == 'noAtts':
            leafLabel = majorityLabel(currentPath)
            completePath = currentPath + [('LABEL', leafLabel)]
            allPaths.append(completePath)
            continue
        splitAtt = findSplitAtt(currentPath, attList)
        values = createBranches(splitAtt)
        for i in range(len(values)):
            value = values[i]
            childPath = currentPath + [(splitAtt, value)]
            pathsToProcess.append(childPath)
    return allPaths
tree = main(df)
for i in range(len(tree)):
    print(tree[i])

[('source', 'US'), ('LABEL', 'No')]
[('source', 'Asia'), ('whiteListedIP', 'Yes'), ('LABEL', 'Yes')]
[('source', 'Asia'), ('whiteListedIP', 'No'), ('packetSize', 'Small'), ('LABEL', 'No')]
[('source', 'Asia'), ('whiteListedIP', 'No'), ('packetSize', 'Medium'), ('appType', 'HTTP'), ('LABEL', 'Yes')]


Question 2:  Perform a feed-forward pass for one iteration using the input values X1 = 0.2 and X2 = 0.5.You should perform your calculations by using the mathematical operators in Python

In [5]:
import math
X1 = 0.2
X2 = 0.5

w13 = 0.05
w14 = -0.01
w15 = 0.02
w23 = 0.01
w24 = 0.03
w25 = -0.01
w36 = 0.01
w46 = 0.05
w56 = 0.0

theta3 = -0.03
theta4 = 0.2
theta5 = 0.05
theta6 = -0.015

In [14]:
def sigmoid(z):
    result = 1 / (1 + math.exp(-z))
    return result

In [15]:
net3 = X1 * w13 + X2 * w23 + theta3
a3 = sigmoid(net3)
print("net3 =", net3)
print("a3 =", a3)

net4 = X1 * w14 + X2 * w24 + theta4
a4 = sigmoid(net4)
print("net4 =", net4)
print("a4 =", a4)

net5 = X1 * w15 + X2 * w25 + theta5
a5 = sigmoid(net5)
print("net5 =", net5)
print("a5 =", a5)

net6 = a3 * w36 + a4 * w46 + a5 * w56 + theta6
a6 = sigmoid(net6)
print("net6 =", net6)
print("a6 =", a6)

net3 = -0.014999999999999996
a3 = 0.496250070310918
net4 = 0.21300000000000002
a4 = 0.553049584279497
net5 = 0.049
a5 = 0.5122475495675138
net6 = 0.017614979917084037
a6 = 0.5044036311138793


Claude AI was used to assist with this project. It helped with the structuring, and debugging the Python code, as well as explaining the underlying logic of the decision tree and neural network implementations. All AI-assisted work was reviewed, tested, and verified to ensure accuracy and understanding.